# B3 — Post-COVID era comparison

Compares retail vs. professional options behaviour around earnings across
three eras, using both the base and extended samples:

| Era | Window | Sample |
|---|---|---|
| Pre-COVID | 2016-01-01 – 2019-12-31 | base |
| COVID | 2020-01-01 – 2021-12-31 | base |
| Post-COVID | 2022-08-01 – 2025-12-31 | extended |

**Requires A1 (base) and A2 (extension) to have been run.**

Three design decisions, all handled inside `covid_era_comparison` so the
cells below stay simple:

1. **The eras live in different physical datasets.** Pre-COVID and COVID come
   from the base `*.parquet` files; post-COVID from the `*_ext` files. The
   comparison functions flip `set_sample` to the correct source per era and
   restore it afterwards — you never switch manually.

2. **Pre-COVID starts in 2016, not 2011.** CBOE's professional-customer
   classification breaks definitionally in 2015 (documented in B2). Starting
   pre-COVID at 2016 keeps the control group comparable across all three eras.

3. **Eras are compared as SEPARATE DiD estimates, never pooled.** A single
   regression spanning a classification break and a pandemic would not be
   interpretable. Three self-contained estimates, compared, is the defensible
   design.

**Interpretation caveat.** Pre-COVID and COVID look alike on most outcomes;
the change is *post*-2021. So this is not a "COVID effect" — frame it as the
post-boom period, not the pandemic. Too much changed at once (zero-commission
maturity, stimulus, meme stocks) to attribute the shift to any single cause.

## Setup

In [1]:
import sys
from pathlib import Path

def find_src_dir(start: Path = None) -> Path:
    current = start or Path.cwd()
    for _ in range(5):
        if (current / "src").exists():
            return current / "src"
        current = current.parent
    raise FileNotFoundError("Could not locate a 'src' folder above the current directory.")

sys.path.append(str(find_src_dir()))

import polars as pl
from paths import DATA_DIR, IBES_DIR, CRSP_DIR

## Three-era difference-in-differences

Binary `treat:post` per outcome, per era, on the balanced panel (the primary
specification). The `sample` column is the audit trail — pre_covid/covid must
read `base`, post_covid must read `ext`.

In [2]:
from analysis.covid_era_comparison import compare_covid_eras

eras = compare_covid_eras(outcomes=["otm", "otm_put", "lt_100", "call", "open"])

print("=== treat:post by era ===")
with pl.Config(tbl_rows=-1, float_precision=4):
    print(eras.pivot(values="coef", index="outcome", on="era"))
    print()
    print(eras)

Sample set to 'base'. Data loads from base files.
Sample set to 'base'. Data loads from base files.
Sample set to 'ext'. Data loads from *_ext files.
Sample set to 'base'. Data loads from base files.
=== treat:post by era ===
shape: (5, 4)
┌─────────┬───────────┬─────────┬────────────┐
│ outcome ┆ pre_covid ┆ covid   ┆ post_covid │
│ ---     ┆ ---       ┆ ---     ┆ ---        │
│ str     ┆ f64       ┆ f64     ┆ f64        │
╞═════════╪═══════════╪═════════╪════════════╡
│ otm     ┆ 0.0412    ┆ 0.0410  ┆ 0.0024     │
│ otm_put ┆ 0.0278    ┆ 0.0195  ┆ 0.0012     │
│ lt_100  ┆ -0.0193   ┆ -0.0129 ┆ 0.0213     │
│ call    ┆ -0.0248   ┆ -0.0130 ┆ -0.0114    │
│ open    ┆ 0.0207    ┆ 0.0187  ┆ 0.0123     │
└─────────┴───────────┴─────────┴────────────┘

shape: (15, 6)
┌────────────┬────────┬─────────┬─────────┬─────────┬───────┐
│ era        ┆ sample ┆ outcome ┆ coef    ┆ p_value ┆ n_obs │
│ ---        ┆ ---    ┆ ---     ┆ ---     ┆ ---     ┆ ---   │
│ str        ┆ str    ┆ str     ┆ f64    

## Size-controlled comparison

The base analysis showed firm size is the dominant confounder — most raw
effects are partly size. This runs the triple interaction with
`log_mktcap * treat * post`, reporting `treat:post` net of the size channel
plus `size_did` (what size itself is doing). An effect that changes across
eras *even after* the size control is a genuine behavioural change, not a
compositional one.

Watch the split: if a large raw effect decomposes into a real `treat:post`
plus a significant `size_did`, size was inflating it; if `size_did` collapses
in an era, the size channel stopped operating there.

In [3]:
from analysis.covid_era_comparison import compare_covid_eras_controlled

ctrl = compare_covid_eras_controlled(outcomes=["otm", "otm_put", "lt_100", "call", "open"])

print("=== size-controlled treat:post by era ===")
with pl.Config(tbl_rows=-1, float_precision=4):
    print(ctrl.pivot(values="treat_post", index="outcome", on="era"))
    print()
    print(ctrl)

Sample set to 'base'. Data loads from base files.
Sample set to 'base'. Data loads from base files.
Sample set to 'ext'. Data loads from *_ext files.
Sample set to 'base'. Data loads from base files.
=== size-controlled treat:post by era ===
shape: (5, 4)
┌─────────┬───────────┬─────────┬────────────┐
│ outcome ┆ pre_covid ┆ covid   ┆ post_covid │
│ ---     ┆ ---       ┆ ---     ┆ ---        │
│ str     ┆ f64       ┆ f64     ┆ f64        │
╞═════════╪═══════════╪═════════╪════════════╡
│ otm     ┆ 0.0245    ┆ 0.0347  ┆ 0.0062     │
│ otm_put ┆ 0.0292    ┆ 0.0232  ┆ -0.0077    │
│ lt_100  ┆ 0.0190    ┆ 0.0218  ┆ 0.0494     │
│ call    ┆ -0.0217   ┆ -0.0133 ┆ 0.0062     │
│ open    ┆ 0.0095    ┆ 0.0107  ┆ 0.0073     │
└─────────┴───────────┴─────────┴────────────┘

shape: (15, 8)
┌────────────┬────────┬─────────┬────────────┬──────────────┬──────────┬────────┬────────┐
│ era        ┆ sample ┆ outcome ┆ treat_post ┆ p_treat_post ┆ size_did ┆ p_size ┆ n_obs  │
│ ---        ┆ ---    ┆ ---  

## Levels — who moves, and is each null/reversal real?

A DiD coefficient near zero can mean "both groups flat" or "both moved
together"; a sign flip needs its cell means checked before it's trusted.
`era_levels_all` shows the four means (retail/procust × baseline/near) per
outcome per era in one table.

Confirm the post-COVID OTM shares sit in the same ~0.6 range as base (not a
broken group), and that the lt_100 and otm_put reversals show sensible cell
movements rather than a data artefact.

In [4]:
from analysis.covid_era_comparison import era_levels_all

lv = era_levels_all(outcomes=["otm", "otm_put", "lt_100", "call", "open"])
with pl.Config(tbl_rows=-1, float_precision=4):
    print(lv.pivot(values="mean_share",
                   index=["outcome", "era", "participant_group"],
                   on="is_near_event").sort("outcome", "era", "participant_group"))

Sample set to 'base'. Data loads from base files.
Sample set to 'base'. Data loads from base files.
Sample set to 'ext'. Data loads from *_ext files.
Sample set to 'base'. Data loads from base files.
Sample set to 'base'. Data loads from base files.
Sample set to 'base'. Data loads from base files.
Sample set to 'ext'. Data loads from *_ext files.
Sample set to 'base'. Data loads from base files.
Sample set to 'base'. Data loads from base files.
Sample set to 'base'. Data loads from base files.
Sample set to 'ext'. Data loads from *_ext files.
Sample set to 'base'. Data loads from base files.
Sample set to 'base'. Data loads from base files.
Sample set to 'base'. Data loads from base files.
Sample set to 'ext'. Data loads from *_ext files.
Sample set to 'base'. Data loads from base files.
Sample set to 'base'. Data loads from base files.
Sample set to 'base'. Data loads from base files.
Sample set to 'ext'. Data loads from *_ext files.
Sample set to 'base'. Data loads from base files.


## Figures

Written to `results/figures/`:

- `fig_era_coefficients.png` — size-controlled treat:post across eras, one
  panel per outcome, filled = significant. The summary figure.
- `fig_era_levels_<outcome>.png` — the four cell means per era as connected
  retail/professional lines, one panel per era, shared y-axis. The OTM one
  shows the retail/professional gap narrowing across eras; the lt_100 one
  shows the position-size reversal.

In [5]:
from analysis.covid_era_figures import build_all_era_figures
build_all_era_figures()

Building era figures...
Sample set to 'base'. Data loads from base files.
Sample set to 'base'. Data loads from base files.
Sample set to 'ext'. Data loads from *_ext files.
Sample set to 'base'. Data loads from base files.
  wrote fig_era_coefficients.png
Sample set to 'base'. Data loads from base files.
Sample set to 'base'. Data loads from base files.
Sample set to 'ext'. Data loads from *_ext files.
Sample set to 'base'. Data loads from base files.
  wrote fig_era_levels_otm.png
Sample set to 'base'. Data loads from base files.
Sample set to 'base'. Data loads from base files.
Sample set to 'ext'. Data loads from *_ext files.
Sample set to 'base'. Data loads from base files.
  wrote fig_era_levels_lt_100.png
Sample set to 'base'. Data loads from base files.
Sample set to 'base'. Data loads from base files.
Sample set to 'ext'. Data loads from *_ext files.
Sample set to 'base'. Data loads from base files.
  wrote fig_era_levels_otm_put.png
Sample set to 'base'. Data loads from base 

### Individual figures

For the thesis, the most communicative are `fig_era_coefficients` (summary),
`fig_era_levels_otm` (the narrowing gap), and `fig_era_levels_lt_100` (the
position-size reversal). Generate any single one directly:

In [6]:
from analysis.covid_era_figures import fig_era_levels, fig_era_coefficients
# fig_era_levels(outcome="otm")
# fig_era_levels(outcome="lt_100")
# fig_era_coefficients()

## Summary of findings

What the three eras show (subject to your run's exact numbers):

- **OTM tilt weakened post-COVID and lost its size-driven component.** Retail
  still shifts toward OTM near earnings (levels confirm it), but the effect is
  smaller and no longer amplified by firm size — connecting to the thesis's
  central finding that size drove much of the apparent effect.
- **Position sizing reversed.** Retail took smaller positions than
  professionals pre/during COVID, larger post-COVID (`lt_100`, survives the
  size control).
- **The put-tilt was pre-2022-specific** (`otm_put` reverses sign).
- **Professional de-risking persists** across all eras (`open`), so the
  professional side is stable while the retail side changed.

The contribution is contextual: the retail options behaviour documented in
the recent literature (built on 2019–2021 data) was period-specific, not a
permanent feature. This does not overturn the base results — it reframes what
period they describe.